# Entrenamiento de pipeline y Gradient Boosting

Construye y guarda dos artefactos: el pipeline de transformación de 23 variables y el modelo GradientBoostingRegressor.

In [ ]:
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler

pd.set_option('display.max_columns', None)

In [ ]:
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'data' / 'raw' / 'train.csv').exists():
    raise FileNotFoundError('Ejecuta el notebook con project-data-input como directorio de trabajo.')

TRAIN_PATH = PROJECT_ROOT / 'data' / 'raw' / 'train.csv'
RENAME_PATH = PROJECT_ROOT / 'scripts' / 'rename_columns.json'
METADATA_PATH = PROJECT_ROOT.parent / 'project-data-processing' / 'scripts' / 'preprocessing_metadata_lowercase.json'
MODELS_PATH = PROJECT_ROOT / 'models'
PIPELINE_PATH = MODELS_PATH / 'Pipeline-Transformacion-Training.joblib'
MODEL_PATH = MODELS_PATH / 'Model-GradientBoostingRegressor.joblib'

with RENAME_PATH.open(encoding='utf-8') as file:
    rename_columns = json.load(file)
with METADATA_PATH.open(encoding='utf-8') as file:
    metadata = json.load(file)

data = pd.read_csv(TRAIN_PATH).rename(columns=rename_columns)
data = data.rename(columns={'SalePrice': 'saleprice'})

feature_order = metadata['minmax_scaler']['feature_order']
TARGET = 'saleprice'
required_columns = [column for column in feature_order if column != 'lotfrontage_na'] + ['yrsold', TARGET]
missing_columns = sorted(set(required_columns) - set(data.columns))
if missing_columns:
    raise ValueError(f'Faltan columnas en train.csv: {missing_columns}')

X_train = data.drop(columns=[TARGET])
y_train = data[TARGET]
print(f'Datos de entrenamiento: {X_train.shape[0]} filas')
print(f'Variables finales: {len(feature_order)}')

In [ ]:
class MetadataFeatureTransformer(BaseEstimator, TransformerMixin):
    """Aplica las reglas de preprocessing_metadata_lowercase.json."""

    QUALITY_MAPPING = {'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5, 'Missing': 0, 'NA': 0}
    FINISH_MAPPING = {'Missing': 0, 'NA': 0, 'Unf': 1, 'LwQ': 2, 'Rec': 3, 'BLQ': 4, 'ALQ': 5, 'GLQ': 6}
    STRING_COLUMNS = ['mssubclass', 'mszoning', 'neighborhood', 'centralair', 'garagetype']
    LOG_COLUMNS = ['lotfrontage', 'firstflrsf', 'grlivarea']
    ADDITIONAL_IMPUTATION = {
        'bsmtfinsf_principal': 0,
        'totalbsmtsf': 0,
        'kitchenqual': 'Missing',
        'garagecars': 0,
        'garagearea': 0,
    }

    def __init__(self, metadata):
        self.metadata = metadata

    def fit(self, X, y=None):
        self.feature_order_ = list(self.metadata['minmax_scaler']['feature_order'])
        return self

    def transform(self, X):
        required = set(self.feature_order_) | {'yrsold'}
        required.discard('lotfrontage_na')
        missing = sorted(required - set(X.columns))
        if missing:
            raise ValueError(f'Faltan columnas para transformar: {missing}')

        transformed = X.copy()
        # Indicador de entrenamiento: debe calcularse antes de imputar lotfrontage.
        transformed['lotfrontage_na'] = transformed['lotfrontage'].isna().astype(int)

        for column, value in self.metadata['frequent_imputation'].items():
            transformed[column] = transformed[column].fillna(value)
        for column, value in self.metadata['numeric_imputation'].items():
            transformed[column] = transformed[column].fillna(value)
        transformed['fireplacequ'] = transformed['fireplacequ'].fillna('Missing')
        for column, value in self.ADDITIONAL_IMPUTATION.items():
            transformed[column] = transformed[column].fillna(value)

        # El nulo de garagetype se representa como texto 'nan', no como Missing.
        transformed['garagetype'] = transformed['garagetype'].where(
            transformed['garagetype'].notna(), 'nan'
        )
        transformed[self.STRING_COLUMNS] = transformed[self.STRING_COLUMNS].astype(str)

        transformed['yearremodadd'] = transformed['yrsold'] - transformed['yearremodadd']
        transformed = transformed.drop(columns=['yrsold'])

        for column in self.LOG_COLUMNS:
            if (transformed[column] <= 0).any():
                raise ValueError(f'{column} contiene valores no positivos; no se puede aplicar np.log.')
            transformed[column] = np.log(transformed[column])

        for column in ['bsmtqual', 'kitchenqual', 'fireplacequ']:
            transformed[column] = transformed[column].map(self.QUALITY_MAPPING)
        transformed['bsmtfintype_principal'] = transformed['bsmtfintype_principal'].map(self.FINISH_MAPPING)

        for column, frequent_labels in self.metadata['rare_labels'].items():
            transformed[column] = transformed[column].where(
                transformed[column].isin(frequent_labels), 'Rare'
            )
        for column, encoding in self.metadata['ordinal_encoding'].items():
            transformed[column] = transformed[column].map(encoding)

        result = transformed.reindex(columns=self.feature_order_)
        if result.isnull().any().any():
            columns_with_nulls = result.columns[result.isnull().any()].tolist()
            raise ValueError(f'Quedaron nulos tras transformar: {columns_with_nulls}')
        return result.astype(float)

    def get_feature_names_out(self, input_features=None):
        return np.asarray(self.feature_order_, dtype=object)

In [ ]:
import sys
scripts_path = str(PROJECT_ROOT / 'scripts')
if scripts_path not in sys.path:
    sys.path.insert(0, scripts_path)
from metadata_transformer import MetadataFeatureTransformer

transform_pipeline = Pipeline([
    ('feature_engineering', MetadataFeatureTransformer(metadata=metadata)),
    ('minmax_scaler', MinMaxScaler()),
])

X_train_transformed = transform_pipeline.fit_transform(X_train, y_train)
if X_train_transformed.shape[1] != len(feature_order):
    raise ValueError(f'Se esperaban {len(feature_order)} columnas y se obtuvieron {X_train_transformed.shape[1]}.')
if not np.isfinite(X_train_transformed).all():
    raise ValueError('El pipeline produjo valores no finitos.')

df_train_features = pd.DataFrame(
    X_train_transformed, index=X_train.index, columns=feature_order
)

model = GradientBoostingRegressor(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=4,
    random_state=42,
)
model.fit(X_train_transformed, y_train)

MODELS_PATH.mkdir(parents=True, exist_ok=True)
joblib.dump(transform_pipeline, PIPELINE_PATH)
joblib.dump(model, MODEL_PATH)

print(f'Pipeline guardado: {PIPELINE_PATH}')
print(f'Modelo guardado: {MODEL_PATH}')
print(f'Modelo entrenado con {model.n_features_in_} variables.')

In [ ]:
# Estas son las 23 columnas transformadas que deben cargarse como features en BigQuery.
df_features_bq = pd.DataFrame({'features': feature_order})
display(df_train_features.head())
display(df_features_bq)
print(df_train_features.columns.tolist())